In [1]:
import os, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt


In [2]:
from torch.utils.data import Dataset as TorchDataset
import json
import os
import torch
import numpy as np
from pathlib import Path

class RawTokenDataset(TorchDataset):
    """ Loads raw uint32 tokens as memmap-backed array """
    def __init__(
        self,
        data_dir,
        window_size,
        stride=1,
        filter_interrupts=True,
        filter_overlaps=False
    ):
        """
        Args:
            data_dir: directory with the same format as `data/train_v0` and `data/val_v0`.
                Notably, has `video.bin` and `metadata.json`
            window_size: number of frames per "video" sequence
            stride: frame skip
            filter_interrupts: Under 3% of training frame sequences are the concatenation of two different clips.
                If filter_interrupts is True, will filter out these sequences using the segment ids.
            filter_overlaps: If False (default), one frame will appear in multiple examples;
                e.g. frame 0 might appear as the first frame in example 0 and also the second frame in example 15.
                If True, will filter out examples so that each frame appears at most once in the dataset.
        """
        data_dir = Path(data_dir)
        with open(data_dir / "metadata.json") as f:
            self.metadata = json.load(f)

        shape = (self.metadata["num_images"], self.metadata["s"], self.metadata["s"])
        video_tokens_path, segment_ids_path, action_tokens_path = [data_dir / f"{name}.bin"
                                                                   for name in ["video", "segment_ids", "actions"]]
        token_dtype = np.dtype(self.metadata.get("token_dtype", "uint32"))
        self.data = np.memmap(video_tokens_path, dtype=token_dtype, mode="r", shape=shape)
        # self.actions = np.memmap(action_tokens_path, dtype=np.uint16, mode="r", shape=(self.metadata["num_images"],))

        if os.path.isfile(segment_ids_path):
            self.segment_ids = np.memmap(
                segment_ids_path,
                dtype=np.int32,
                mode="r",
                shape=(self.metadata["num_images"],)
            )
        else:
            self.segment_ids = None
            if filter_interrupts:
                raise NotImplementedError("Cannot filter interrupted sequences without segment ids.")

        self.window_size, self.stride = window_size, stride
        # Number of frames between the first and last frames of a video sequence (excluding one endpoint frame)
        self.video_len = (self.window_size - 1) * self.stride

        self.valid_start_inds = []
        for start_ind in range(len(self.data) - self.video_len):
            # Assuming `segment_ids` is monotonically increasing, a sequence is interrupted
            # if the first and last frames have different segment ids.
            if not (filter_interrupts and self.segment_ids[start_ind] != self.segment_ids[start_ind + self.video_len]):
                self.valid_start_inds.append(start_ind)

        if filter_overlaps:
            # Instead of using a sliding window, use each frame at most once
            filtered_start_inds = []
            for start_ind in self.valid_start_inds:
                overlapping_start_inds = {start_ind - i * self.stride for i in range(1, self.window_size)}
                # all sequences from `overlapping_start_inds` will also contain `start_ind`,
                # so exclude sequence starting from `start_ind` if any of `overlapping_start_inds` is already being used
                for existing_start_ind in filtered_start_inds[-self.window_size * self.stride:]:
                    # Bound could be improved
                    if existing_start_ind in overlapping_start_inds:
                        break
                else:
                    filtered_start_inds.append(start_ind)

            self.valid_start_inds = filtered_start_inds

    def __len__(self):
        return len(self.valid_start_inds)

    def __getitem__(self, idx):
        """
        Returns a flattened sequence of tokens representing `self.window_size` frames,
        spaced `self.stride` apart.
        """
        start_ind = self.valid_start_inds[idx]
        x = torch.from_numpy((self.data[start_ind : start_ind + self.video_len + 1 : self.stride]).astype(np.int64))
        x = x.flatten()

        attention_mask = torch.ones_like(x)
        return {
            "input_ids": x,
            "labels": x,
            "attention_mask": attention_mask,
        }

In [3]:
class Cfg:
    # data
    data_dir = None     # 例: "./data/train_v0"
    window_size = 6
    stride = 1

    # slots
    num_slots = 8       # 「二分離」なら 2。増やしたければ 11などに
    d_model = 128
    slot_iters = 5

    # decoder
    dec_hidden = 256

    # training
    batch_size = 8
    num_epochs = 20
    lr = 3e-4
    weight_decay = 1e-4
    grad_clip = 1.0
    device = "cuda" if torch.cuda.is_available() else "cpu"

cfg = Cfg()

def infer_vocab_size(dataset, n_batches=50, batch_size=64, device="cpu"):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=False)
    mx = 0
    for i, batch in enumerate(loader):
        x = batch["input_ids"]
        mx = max(mx, int(x.max().item()))
        if i+1 >= n_batches:
            break
    return mx + 1


In [4]:
class SlotAttention(nn.Module):
    def __init__(self, num_slots, dim, iters=3):
        super().__init__()
        self.num_slots = num_slots
        self.iters = iters
        self.scale = dim ** -0.5

        self.slots_mu = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.slots_logsigma = nn.Parameter(torch.zeros(1, 1, dim))

        self.norm_in = nn.LayerNorm(dim)
        self.norm_slots = nn.LayerNorm(dim)
        self.norm_ff = nn.LayerNorm(dim)

        self.to_q = nn.Linear(dim, dim, bias=False)
        self.to_k = nn.Linear(dim, dim, bias=False)
        self.to_v = nn.Linear(dim, dim, bias=False)

        self.gru = nn.GRUCell(dim, dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim*2), nn.GELU(), nn.Linear(dim*2, dim)
        )

    def forward(self, inputs, prev_slots=None):
        # inputs: [B, N, D]
        B, N, D = inputs.shape

        if prev_slots is None:
            mu = self.slots_mu.expand(B, self.num_slots, -1)
            sigma = self.slots_logsigma.expand(B, self.num_slots, -1).exp()
            slots = mu + sigma * torch.randn_like(mu)
        else:
            slots = prev_slots

        x = self.norm_in(inputs)
        k = self.to_k(x)  # [B,N,D]
        v = self.to_v(x)

        for _ in range(self.iters):
            slots_prev = slots
            s = self.norm_slots(slots)
            q = self.to_q(s)  # [B,K,D]

            dots = torch.einsum("bkd,bnd->bkn", q, k) * self.scale  # [B,K,N]
            attn = dots.softmax(dim=1) + 1e-8                       # slot方向
            attn = attn / attn.sum(dim=-1, keepdim=True)            # N方向に正規化
            updates = torch.einsum("bnd,bkn->bkd", v, attn)         # [B,K,D]

            slots = self.gru(
                updates.reshape(-1, D),
                slots_prev.reshape(-1, D)
            ).reshape(B, self.num_slots, D)

            slots = slots + self.mlp(self.norm_ff(slots))

        return slots


class TokenSlotDecoder(nn.Module):
    """
    slots -> (mask_logits, token_logits)
    mask_logits: [B, K, N]  (softmax over K)
    token_logits: [B, K, N, V] (softmax over V)
    """
    def __init__(self, num_slots, d_model, n_positions, vocab_size, hidden=256):
        super().__init__()
        self.num_slots = num_slots
        self.n_positions = n_positions
        self.vocab_size = vocab_size

        # learned per-position embedding (like spatial broadcast but 1D positions)
        self.pos = nn.Parameter(torch.randn(1, 1, n_positions, d_model) * 0.02)

        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden), nn.GELU(),
            nn.Linear(hidden, d_model), nn.GELU(),
        )
        self.mask_head = nn.Linear(d_model, 1)
        self.token_head = nn.Linear(d_model, vocab_size)

    def forward(self, slots):
        # slots: [B, K, D]
        B, K, D = slots.shape
        N = self.n_positions
        h = slots[:, :, None, :].expand(B, K, N, D) + self.pos  # [B,K,N,D]
        h = self.mlp(h)
        mask_logits = self.mask_head(h).squeeze(-1)             # [B,K,N]
        token_logits = self.token_head(h)                       # [B,K,N,V]
        return mask_logits, token_logits


In [5]:
class DiscreteTokenSlotModel(nn.Module):
    """
    入力: tokens [B, T, N] (N=s*s)
    1フレームずつ SlotAttention で K slot に分解（任意で prev_slots を渡して追跡）
    デコーダで (mask_logits, token_logits) を出し、混合カテゴリの対数尤度で学習
    """
    def __init__(self, s, window_size, vocab_size, num_slots=2, d_model=128, slot_iters=5, dec_hidden=256):
        super().__init__()
        self.s = s
        self.T = window_size
        self.N = s * s
        self.V = vocab_size
        self.K = num_slots
        self.D = d_model

        self.tok_emb = nn.Embedding(vocab_size, d_model)

        # factorized positional embeddings
        self.pos_time = nn.Parameter(torch.randn(1, window_size, 1, d_model) * 0.02)
        self.pos_space = nn.Parameter(torch.randn(1, 1, self.N, d_model) * 0.02)

        self.slot_attn = SlotAttention(num_slots=num_slots, dim=d_model, iters=slot_iters)
        self.decoder = TokenSlotDecoder(num_slots=num_slots, d_model=d_model, n_positions=self.N,
                                        vocab_size=vocab_size, hidden=dec_hidden)

    def forward(self, tokens):
        """
        tokens: [B,T,N] long
        returns:
          nll: scalar
          masks: [B,T,K,N] soft (mixing weights)
          hard_assign: [B,T,N] argmax slot
        """
        B, T, N = tokens.shape
        assert T == self.T and N == self.N

        # embed inputs
        x = self.tok_emb(tokens)                                # [B,T,N,D]
        x = x + self.pos_time[:, :T] + self.pos_space           # [B,T,N,D]

        all_masks = []
        all_hard = []
        nll_total = 0.0

        prev_slots = None
        for t in range(T):
            inp = x[:, t]                                       # [B,N,D]
            slots = self.slot_attn(inp, prev_slots=prev_slots)  # [B,K,D]
            prev_slots = slots

            mask_logits, token_logits = self.decoder(slots)     # [B,K,N], [B,K,N,V]

            # mixing weights (log space)
            log_pi = torch.log_softmax(mask_logits, dim=1)      # [B,K,N]

            # per-slot token log prob of the ground-truth token
            # log p_k(x): [B,K,N]
            log_pk = torch.log_softmax(token_logits, dim=-1)
            y = tokens[:, t]                                    # [B,N]
            log_pk_y = log_pk.gather(dim=-1, index=y[:, None, :, None].expand(B, self.K, N, 1)).squeeze(-1)

            # mixture log prob: log sum_k exp(log_pi + log_pk_y)
            logp = torch.logsumexp(log_pi + log_pk_y, dim=1)    # [B,N]
            nll = -logp.mean()
            nll_total = nll_total + nll

            masks = torch.softmax(mask_logits, dim=1)           # [B,K,N]
            hard = torch.argmax(masks, dim=1)                   # [B,N]

            all_masks.append(masks)
            all_hard.append(hard)

        nll_total = nll_total / T
        masks = torch.stack(all_masks, dim=1)  # [B,T,K,N]
        hard = torch.stack(all_hard, dim=1)   # [B,T,N]
        return nll_total, masks, hard


In [6]:
@torch.no_grad()
def visualize_assignments(hard_assign, s, max_frames=6):
    """
    hard_assign: [T, N] (N=s*s) int
    """
    T, N = hard_assign.shape
    Tshow = min(T, max_frames)
    plt.figure(figsize=(2*Tshow, 2))
    for t in range(Tshow):
        plt.subplot(1, Tshow, t+1)
        img = hard_assign[t].view(s, s).cpu().numpy()
        plt.imshow(img)  # colormap default
        plt.axis("off")
        plt.title(f"t={t}")
    plt.show()


In [9]:
# ---- 1) dataset ----
cfg.data_dir = "/root/work/data/raw/train_v1.1"  # ★ここを自分のパスに
dataset = RawTokenDataset(
    data_dir=cfg.data_dir,
    window_size=cfg.window_size,
    stride=cfg.stride,
    filter_interrupts=True,
    filter_overlaps=False,
)

s = dataset.metadata["s"]
T = dataset.window_size
N = s * s

# ---- 2) vocab size ----
vocab_size = dataset.metadata.get("vocab_size", None)
if vocab_size is None:
    print("[Info] metadataに vocab_size が無いので推定します（サブサンプル）...")
    vocab_size = infer_vocab_size(dataset, n_batches=50, batch_size=cfg.batch_size)
print("s =", s, "T =", T, "N =", N, "vocab_size =", vocab_size)

# ---- 3) loader ----
loader = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=True, drop_last=True, num_workers=0)

# ---- 4) model ----
model = DiscreteTokenSlotModel(
    s=s,
    window_size=T,
    vocab_size=vocab_size,
    num_slots=cfg.num_slots,
    d_model=cfg.d_model,
    slot_iters=cfg.slot_iters,
    dec_hidden=cfg.dec_hidden,
).to(cfg.device)

opt = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

print("params:", sum(p.numel() for p in model.parameters())/1e6, "M")

# ---- 5) train ----
history = []
for epoch in range(cfg.num_epochs):
    model.train()
    total = 0.0

    pbar = tqdm(loader, desc=f"epoch {epoch+1}/{cfg.num_epochs}")
    for batch in pbar:
        x = batch["input_ids"].to(cfg.device)          # [B, L]
        # reshape back to [B,T,N]
        x = x.view(cfg.batch_size, T, N)

        opt.zero_grad(set_to_none=True)
        loss, masks, hard = model(x)
        loss.backward()
        if cfg.grad_clip is not None and cfg.grad_clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        opt.step()

        total += float(loss.item())
        pbar.set_postfix(nll=f"{loss.item():.4f}")

    avg = total / len(loader)
    history.append(avg)
    print(f"Epoch {epoch+1}: avg NLL = {avg:.4f}")

    # 簡易可視化（最初のバッチの先頭サンプル）
    if (epoch + 1) % 5 == 0:
        model.eval()
        with torch.no_grad():
            # 直近バッチから表示
            hard0 = hard[0].detach().cpu()   # [T,N]
            visualize_assignments(hard0, s=s, max_frames=T)

plt.figure()
plt.plot(history, marker="o")
plt.grid(True)
plt.title("Train avg NLL")
plt.show()


s = 16 T = 6 N = 256 vocab_size = 262144
params: 67.718529 M


epoch 1/20:   0%|          | 0/167958 [01:01<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 128.00 GiB. GPU 0 has a total capacity of 15.99 GiB of which 13.88 GiB is free. Of the allocated memory 823.38 MiB is allocated by PyTorch, and 10.62 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)